In [1]:
import os
import numpy as np
import tensorflow as tf
import cv2 as cv
from tqdm import tqdm
import matplotlib.pyplot as plt
from keras.applications.vgg16 import VGG16, preprocess_input
import tensorflow.keras.layers as tfl
from sklearn import metrics
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

In [2]:
def loadData(dir_path):
    X = []
    y = []
    labels = dict()
    i = 0
    for path in tqdm(sorted(os.listdir(dir_path))): #traverse through directories under Train/Val/Test
        if not path.startswith('.'):
            labels[i] = path
            for file in os.listdir(dir_path+'\\'+path):
                if not file.startswith('.'):
                    img = cv.imread(dir_path+'\\'+path+'\\'+file)
                    X.append(img)
                    y.append(i)
            i=i+1
    X = np.array(X,dtype='object')
    y = np.array(y)
    print(f'{len(X)} images loaded from {dir_path} directory.')
    return X,y,labels


In [3]:
X_train, y_train, labels = loadData(r'C:\Users\sriha\OneDrive\Desktop\Projects\PD_Project\TRAIN')
X_val, y_val, _ = loadData(r'C:\Users\sriha\OneDrive\Desktop\Projects\PD_Project\VAL')
X_test, y_test, _ = loadData(r'C:\Users\sriha\OneDrive\Desktop\Projects\PD_Project\TEST')
IMG_SIZE = (64,64)

100%|██████████| 2/2 [00:12<00:00,  6.11s/it]


559 images loaded from C:\Users\sriha\OneDrive\Desktop\Projects\PD_Project\TRAIN directory.


100%|██████████| 2/2 [00:01<00:00,  1.40it/s]


64 images loaded from C:\Users\sriha\OneDrive\Desktop\Projects\PD_Project\VAL directory.


100%|██████████| 2/2 [00:00<00:00,  3.51it/s]

65 images loaded from C:\Users\sriha\OneDrive\Desktop\Projects\PD_Project\TEST directory.


In [4]:
def preprocess_imgs(set, img_size):
    set_new = []
    for img in set:
        img = cv.resize(img, dsize=img_size, interpolation=cv.INTER_CUBIC)
        set_new.append(preprocess_input(img))
    return np.array(set_new)

In [5]:
X_train_fin = preprocess_imgs(set=X_train,img_size=IMG_SIZE)
X_val_fin = preprocess_imgs(set=X_val,img_size=IMG_SIZE)
X_test_fin = preprocess_imgs(set=X_test,img_size=IMG_SIZE)

In [8]:
rand = RandomForestClassifier(n_estimators=500,random_state=0)
X_train_rf = X_train_fin.reshape((X_train_fin.shape[0],-1))
rand.fit(X_train_rf,y_train)

RandomForestClassifier(n_estimators=500, random_state=0)

In [ ]:
X_val_rf = X_val_fin.reshape((X_val_fin.shape[0],-1))
X_test_rf = X_test_fin.reshape((X_test_fin.shape[0],-1))

In [ ]:
yhat_val_rf_ini = rand.predict(X_val_rf)
yhat_test_rf_ini = rand.predict(X_test_rf)
yhat_val_rf = [1 if x>0.5 else 0 for x in yhat_val_rf_ini]
yhat_test_rf = [1 if x>0.5 else 0 for x in yhat_test_rf_ini]

In [ ]:
confusion_matrix_val = metrics.confusion_matrix(y_val,yhat_val_rf)
cmv = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix_val, display_labels = [False, True])
print(f'F1 measure: {metrics.f1_score(y_val,yhat_val_rf)}\nROC AUC: {metrics.roc_auc_score(y_val,yhat_val_rf)}')
cmv.plot()
plt.show()

In [ ]:
confusion_matrix_test = metrics.confusion_matrix(y_test,yhat_test_rf)
cmt = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix_test, display_labels = [False, True])
print(f'F1 measure: {metrics.f1_score(y_test,yhat_test_rf)}\nROC AUC: {metrics.roc_auc_score(y_test,yhat_test_rf)}')
cmt.plot()
plt.show()